# **Dự Đoán Chất Lượng Không Khí Cho Ngày Hôm Sau Ở TP.HCM**

Chất lượng không khí tại TP. Hồ Chí Minh ngày càng trở thành mối quan tâm lớn của người dân, đặc biệt với chỉ số AQI trung bình năm 2022-2026 đạt 81.6 (mức Trung bình theo thang US), trong đó nhiều ngày vượt ngưỡng 100 - mức Không tốt cho nhóm nhạy cảm. Khả năng dự báo AQI trước một ngày giúp người dân lên kế hoạch sinh hoạt, hạn chế phơi nhiễm, và hỗ trợ các cơ quan quản lý môi trường.

### **Mục tiêu notebook này:**

- Tính toán SHAP values cho Best Model đã chọn ở phần Regression.
- Vẽ Summary Plot để xác định feature nào quan trọng nhất đối với toàn bộ tập dữ liệu.
- Vẽ Beeswarm Plot để hiểu chiều hướng tác động (giá trị feature cao/thấp đẩy AQI lên hay xuống).
- Vẽ Waterfall Plot để giải thích chi tiết một dự đoán cụ thể.
- Vẽ Dependence Plot để xem mối quan hệ tương tác giữa 2 features.
- So sánh kết quả SHAP với Feature Importance đã tính ở phần Regression để đối chiếu.

## **00. Import và cấu hình**

In [ ]:
# Standard Libraries
import warnings
import gdown
import os
import pickle
import joblib
import json
import copy

from google.colab import drive
from datetime import datetime

# Data Manipulation - Math
import numpy as np
import pandas as pd

# Data Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import folium
from matplotlib.colors import LinearSegmentedColormap
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.stats import gaussian_kde
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colorbar import ColorbarBase

# Machine Learning - Preprocessing
import shap

In [ ]:
# Cấu hình
warnings.filterwarnings(
    "ignore"
)  # Tắt các cảnh báo không cần thiết cho notebook sạch hơn
pd.set_option(
    "display.float_format", "{:.2f}".format
)  # Hiển thị số thập phân gọn hơn (làm tròn 2 chữ số thập phân)
plt.rcParams.update(
    {
        "figure.dpi": 150,
        "axes.grid": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.family": "DejaVu Sans",
    }
)  # Cấu hình style mặc định cho toàn bộ biểu đồ matplotlib trong notebook

# AQI Color Palette
# Đây là màu chính thức của US EPA, để đảm bảo nhất quán toàn bộ notebook
AQI_COLORS = {
    "Good": "#00E400",
    "Moderate": "#FFFF00",
    "Unhealthy for Sensitive Groups": "#FF7E00",
    "Unhealthy": "#FF0000",
    "Very Unhealthy": "#8F3F97",
    "Hazardous": "#7E0023",
}

AQI_BINS = [0, 50, 100, 150, 200, 300, 500]
AQI_LABELS = list(AQI_COLORS.keys())
AQI_MAPPING = {label: idx for idx, label in enumerate(AQI_LABELS)}


# Hàm AQI_CATEGORY()
def AQI_CATEGORY(value):
    """
    Phân loại mức AQI theo thang US EPA
    Input : Giá trị AQI (số thực)
    Output: Tên mức (string)
    """
    if value <= 50:
        return "Good"
    elif value <= 100:
        return "Moderate"
    elif value <= 150:
        return "Unhealthy for Sensitive Groups"
    elif value <= 200:
        return "Unhealthy"
    elif value <= 300:
        return "Very Unhealthy"
    else:
        return "Hazardous"


# Tạo Custom Gradient Colormap từ AQI_COLORS và AQI_BINS
# Chuẩn hóa các mốc AQI vì LinearSegmentedColormap là thang đo 0.0 đến 1.0
MAX_AQI = 500
positions = [
    val / MAX_AQI for val in AQI_BINS
]  # positions sẽ trở thành [0.0, 0.1, 0.2, 0.3, 0.4, 0.6, 1.0]

# Vì AQI_BINS có 7 mốc nhưng AQI_COLORS chỉ có 6 màu
colors = list(AQI_COLORS.values())
gradient_colors = colors + [colors[-1]]  # Nhân đôi màu cuối cùng

# Ghép vị trí và màu sắc lại để tạo dải gradient
color_mapping = list(
    zip(positions, gradient_colors)
)  # Gán các màu tương ứng với từng giá trị
AQI_gradient_cmap = LinearSegmentedColormap.from_list(
    "AQI_gradient", color_mapping
)  # Tạo ra dãy màu liên tục thay vì riêng lẻ

In [ ]:
# drive.mount('/content/drive')
ROOT = "/content/drive/MyDrive/HCMUS/Nhập Môn Khoa Học Dữ Liệu/Mini Project"

figures_dir = os.path.join(ROOT, "figures")
os.makedirs(figures_dir, exist_ok=True)

In [ ]:
folder_ID = "1b8LGMtOwMrj6FDMODA8-rJNq0cKXlgBA"
folder_url = f"https://drive.google.com/drive/folders/{folder_ID}"

models_ID = "1N7bZA7lbf-DW2k-urlCpw967sUT3aKu-"
models_url = f"https://drive.google.com/drive/folders/{models_ID}"

gdown.download_folder(folder_url, output="data", quiet=False)
gdown.download_folder(models_url, output="models", quiet=False)

X_train = pd.read_csv("data/X_train.csv", index_col="date", parse_dates=True)
X_test = pd.read_csv("data/X_test.csv", index_col="date", parse_dates=True)
y_train = pd.read_csv("data/y_train.csv", index_col="date", parse_dates=True)
y_test = pd.read_csv("data/y_test.csv", index_col="date", parse_dates=True)

best_model = joblib.load("models/best_regressor.pkl")
feature_cols = joblib.load("models/feature_cols.pkl")

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(
    f"Khoảng thời gian Test: {X_test.index.min().date()} → {X_test.index.max().date()}"
)

**Lưu ý quan trọng:** SHAP values được tính trên tập **Test** (không phải Train) vì mục đích là giải thích cách model dự đoán trên dữ liệu chưa từng thấy - phản ánh đúng hành vi thực tế khi triển khai `predict.py`.

## **05. SHAP Explainability**



**SHAP (SHapley Additive exPlanations)** là phương pháp giải thích mô hình dựa trên lý thuyết trò chơi hợp tác (cooperative game theory) - cụ thể là giá trị Shapley. **Ý tưởng cốt lõi:** Mỗi feature được xem như một "người chơi" đóng góp vào kết quả dự đoán, và SHAP value đo lường chính xác mức đóng góp đó.

**Công thức trực quan:**

$$\text{Dự đoán} = \text{Base value} + \sum_{i=1}^{n} \text{SHAP}_i$$

Trong đó:
- **Base value**: Giá trị dự đoán trung bình nếu không biết bất kỳ thông tin gì (tương đương AQI trung bình của tập Train).
- **SHAP$_i$**: Mức đóng góp của feature thứ $i$ - dương nghĩa là đẩy dự đoán lên cao hơn Base value, âm nghĩa là kéo xuống thấp hơn Base value.

**Tại sao dùng SHAP thay vì chỉ dùng Feature Importance (Split/Gain)?**
- Feature Importance chỉ cho biết feature nào **quan trọng**, không cho biết **quan trọng theo chiều nào** (tăng hay giảm AQI).
- SHAP giải thích được **từng dự đoán cụ thể**, không chỉ tổng quan toàn bộ model.
- SHAP có nền tảng lý thuyết toán học chặt chẽ (tính nhất quán - consistency), không bị thiên vị bởi loại feature (liên tục vs nhị phân) như Split Importance.

### **5.1. SHAP Values**

In [ ]:
# TreeExplainer - tối ưu cho các mô hình dựa trên cây (XGBoost, LightGBM, Random Forest)
# Nếu Best Model là Ridge (Linear), cần dùng shap.LinearExplainer thay thế

model_type = type(best_model).__name__

if "Ridge" in model_type or "Linear" in model_type:
    print(f"Best Model là {model_type} (Linear) - dùng shap.LinearExplainer")
    masker = shap.maskers.Independent(X_train, max_samples=X_train.shape[0])
    explainer = shap.LinearExplainer(best_model, masker)
else:
    print(f"Best Model là {model_type} (Tree-based) - dùng shap.TreeExplainer")
    explainer = shap.TreeExplainer(best_model)

# Tính SHAP values trên tập Test
shap_values = explainer(X_test)

print(f"   Shape: {shap_values.values.shape}")
print(
    f"   Base value: {shap_values.base_values[0] if hasattr(shap_values.base_values, '__len__') else shap_values.base_values:.2f}"
)

#### **Nhận xét chung:**

- Base value xấp xỉ AQI trung bình của tập Train (~78.75) - đây là điểm cơ sở trước khi xét đến đặc điểm riêng của từng ngày.
- Mỗi dòng trong `shap_values.values` tương ứng với 1 ngày trong tập Test, mỗi cột là mức đóng góp của 1 feature cho ngày đó.

### **5.2. Summary Plot - Tổng quan tầm quan trọng tổng thể**

Summary Plot (dạng bar) xếp hạng các features theo **Mean |SHAP Value|** - trung bình giá trị tuyệt đối của SHAP trên toàn bộ tập Test. Feature có thanh dài nhất là feature ảnh hưởng nhiều nhất đến dự đoán, dù đó là ảnh hưởng tăng hay giảm AQI.

In [ ]:
best_model_name = "Ridge Regression"

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, plot_type="bar", max_display=20, show=False)
plt.title(
    f"SHAP Summary Plot - {best_model_name}\nTop 20 Features Quan Trọng Nhất",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "SHAP_summary_plot")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

#### **Nhận xét Summary Plot:**

- **Feature đứng đầu:** `pm2_5` với mức độ ảnh hưởng trung bình (mean |SHAP value|) hơn 17.5. Kết quả này hoàn toàn khớp với kỳ vọng của nhóm. Giá trị SHAP của pm2_5 vượt trội hoàn toàn so với các biến còn lại (gấp hơn 3 lần so với biến đứng thứ hai). Điều này cho thấy nồng độ PM2.5 hiện tại có sức nặng quyết định lớn nhất đến kết quả dự đoán của mô hình Ridge Regression.

- **So sánh top 5 features của SHAP với Feature Importance của các mô hình hồi quy:**

| | SHAP | Ridge | RF | DT | LGBM | XGB |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | pm2_5 | pm2_5 | pm2_5 | pm2_5 | pm2_5 | pm2_5 |
| 2 | diff_ 1 |  diff_ 1 | pm10 | rolling_std_30 | diff_ 1 | diff_ 1 |
| 3 | nitrogen_dioxide | ozone | nitrogen_dioxide | carbon_monoxide | diff_ 7 | diff_ 7 |
| 4 | ozone | nitrogen_dioxide | carbon_monoxide | pm10 | carbon_monoxide | carbon_monoxide |
| 5 | t-1 | t-1 | t-1 | sulphur_dioxide | ozone | ozone |

> **Kết luận:** Nếu cần chọn ra một tập Core Features để làm gọn mô hình mà không làm giảm hiệu suất, nhóm sẽ chọn:
> - Biến cốt lõi: pm2_5 (Bắt buộc).
> - Biến động lượng (Trend/Lag): diff_1 (xuất hiện ở 4/6 mô hình), diff_7 (quan trọng với Tree mạnh), và t-1.
> - Biến khí tượng / hóa học: ozone, carbon_monoxide, và nitrogen_dioxide. Các mô hình khác nhau sẽ tự biết cách tận dụng một trong số các loại khí này để tối ưu hóa kết quả.

### **5.3. Beeswarm - Chiều hướng tác động**

Khác với Summary Plot dạng bar (chỉ cho biết "quan trọng bao nhiêu"), Beeswarm Plot cho biết thêm **chiều hướng tác động**:
- Mỗi chấm là 1 ngày trong tập Test.
- Vị trí trên trục X: Giá trị SHAP của ngày đó (âm = kéo AQI xuống, dương = đẩy AQI lên).
- Màu sắc: Giá trị thực tế của feature đó (đỏ = giá trị cao, xanh = giá trị thấp).

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, max_display=20, show=False)
plt.title(
    f"SHAP Beeswarm Plot - {best_model_name}\nĐỏ = Giá Trị Feature Cao | Xanh = Giá Trị Feature Thấp",
    fontsize=12,
    fontweight="bold",
)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "SHAP_beeswarm_plot")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

#### **Nhận xét Beeswarm Plot:**

- **Đối với feature t-1 (và feature đứng đầu pm2_5):** Nồng độ pm2_5 hiện tại mới là yếu tố trực tiếp đẩy dự đoán AQI lên cao (chấm đỏ bên phải). Trong khi đó, trong bối cảnh đã biết pm2_5, nếu t-1 cao thì mô hình sẽ có xu hướng hãm kết quả dự đoán lại một chút.

- **Đối với feature is_dry_season:** Ở hàng is_dry_season, các chấm màu đỏ (giá trị = 1, tức là mùa khô) lại tụ tập ở bên trái (SHAP value âm). Các chấm xanh (mùa mưa) nằm ở mức 0 hoặc hơi dương nhẹ. Điều này có vẻ trái ngược với những phát hiện từ EDA (thường mùa khô sẽ có AQI cao hơn do không có mưa rửa trôi bụi). Tuy nhiên, đây là sự khác biệt giữa tác động biên (EDA) và tác động cục bộ (Mô hình). Sau khi mô hình đã biết rõ các thông số nồng độ bụi (pm2_5, pm10) và khí độc (Ozone, NO2...) của ngày hôm đó, thì việc dán nhãn "ngày này thuộc mùa khô" không làm tăng thêm mức độ ô nhiễm nữa, mà thậm chí đóng vai trò như một hệ số bù trừ làm giảm nhẹ kết quả xuống.

> **Kết luận:** Biểu đồ SHAP của mô hình Ridge cho thấy rõ đặc tính tác động cục bộ. Nó không thể hiện mối quan hệ tuyệt đối như EDA, mà thể hiện: "Khi giữ nguyên tất cả các thông số khác (đặc biệt là nồng độ các chất ô nhiễm), biến này sẽ tác động thêm vào dự đoán như thế nào?".

### **5.4. Waterfall Plot - Giải thích 1 dự đoán cụ thể**

Waterfall Plot trả lời câu hỏi cụ thể: **"Tại sao mô hình dự đoán AQI = X cho ngày này?"** - bằng cách hiển thị từng feature đóng góp bao nhiêu, cộng dồn từ Base value đến giá trị dự đoán cuối cùng.

Ở đây nhóm chọn ngày có **AQI dự đoán cao nhất** trong tập Test để minh họa.

In [ ]:
# Tìm ngày có AQI dự đoán cao nhất trong tập Test
y_pred_test = np.clip(best_model.predict(X_test), 0, 500)
idx_max = int(np.argmax(y_pred_test))
date_max = X_test.index[idx_max]

actual_aqi = float(np.ravel(y_test.iloc[idx_max])[0])
predicted_aqi = float(np.ravel(y_pred_test[idx_max])[0])

print(
    f"Ngày được chọn để giải thích: {date_max.strftime('%d/%m/%Y') if hasattr(date_max, 'strftime') else date_max}"
)
print(f"AQI thực tế   : {actual_aqi:.1f}")
print(f"AQI dự đoán   : {predicted_aqi:.1f}")

In [ ]:
plt.figure(figsize=(10, 7))
shap.waterfall_plot(shap_values[idx_max], max_display=14, show=False)
date_label = (
    date_max.strftime("%d/%m/%Y") if hasattr(date_max, "strftime") else str(date_max)
)
plt.title(
    f"SHAP Waterfall Plot - Ngày {date_label}\nAQI Dự Đoán = {y_pred_test[idx_max]:.0f}",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "SHAP_waterfall_plot")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

#### **Phân tích Waterfall Plot**

Ngày [19/06/2025]: AQI dự đoán = [155.506]
- Bắt đầu từ Base value = [78.745] (AQI trung bình tập Train)
- `pm2_5` = [3.41] → [+84.59] (Nồng độ bụi mịn trong ngày rất cao, đây là yếu tố chính đẩy dự đoán AQI vọt lên mức ô nhiễm).
- `diff_1` = [3.05] → [-18.65] (Biến động lượng đóng vai trò hãm mạnh nhất để dự đoán không bị quá lố).
- `nitrogen_dioxide` = [1.603] → [+8.61] (Nồng độ khí NO2 cao tiếp tục đẩy mức độ ô nhiễm lên).
- `diff_7` = [1.797] → [-7.61] (Biến động lượng chu kỳ 7 ngày giúp điều chỉnh giảm).
- `t-1` = [1.028] → [-4.38] (Mô hình sử dụng AQI hôm qua như một hệ số bù trừ nhẹ, kéo giảm dự đoán xuống).
- Các features như is_dry_season, rolling_mean_7... có mức độ ảnh hưởng rất thấp trong ngày này, được gộp chung vào nhóm "27 other features" với tổng tác động là -0.1.
- Tổng cộng → Dự đoán cuối cùng = [155.506].

> Đây là cách mô hình hoạt động.

### **5.5. Dependence Plot - Mối quan hệ giữa 2 Features**

Dependence Plot cho thấy mối quan hệ giữa **giá trị của 1 feature** và **SHAP value tương ứng** của nó, đồng thời tô màu theo 1 feature thứ 2 để phát hiện tương tác (interaction) giữa 2 features.

Ở đây nhóm xem xét feature quan trọng nhất và tương tác của nó với `is_dry_season`.

In [ ]:
# Xác định feature quan trọng nhất để vẽ Dependence Plot
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
top_feature = X_test.columns[np.argmax(mean_abs_shap)]
print(f"Feature quan trọng nhất: {top_feature}")

# Feature thứ 2 để tô màu tương tác - ưu tiên is_dry_season nếu có
interaction_feature = "is_dry_season" if "is_dry_season" in X_test.columns else None

In [ ]:
plt.figure(figsize=(10, 6))
shap.dependence_plot(
    top_feature,
    shap_values.values,
    X_test,
    interaction_index=interaction_feature,
    show=False,
)
plt.title(
    f"SHAP Dependence Plot x {top_feature}"
    + (f" x {interaction_feature}" if interaction_feature else ""),
    fontsize=12,
    fontweight="bold",
)
plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "SHAP_dependence_plot")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

#### **Nhận xét Dependence Plot:**

- **Xu hướng tổng thể:** Có mối quan hệ đồng biến hoàn hảo (tuyến tính). Khi giá trị thực tế của pm2_5 tăng lên, mức độ đóng góp (SHAP value) của nó vào kết quả dự đoán cũng tăng theo tỷ lệ thuận. Các điểm dữ liệu xếp thành một đường thẳng tắp, minh chứng rõ nét cho bản chất phương trình bậc nhất của mô hình Ridge Regression.
- **Tương tác với is_dry_season:** Hoàn toàn không có sự tương tác giữa 2 biến này trong mô hình hiện tại. Tại bất kỳ một mức nồng độ pm2_5 nào (cùng một giá trị trục X), các điểm màu đỏ (mùa khô) và màu xanh (mùa mưa) đều nằm đè lên cùng một đường thẳng duy nhất, không hề có sự phân tách hay chênh lệch độ cao. Lý do là vì Ridge Regression là mô hình cộng tính độc lập. Nếu không tự tạo ra một biến tương tác nhân (ví dụ: pm2_5 * is_dry_season) ở bước tiền xử lý, mô hình sẽ mặc định tác động của bụi là độc lập và không bị thay đổi bởi yếu tố thời tiết.

### **5.6. So sánh SHAP với Feature Importance (Split/Gain)**


Ở phần huấn luyện mô hình đã tính Feature Importance theo phương pháp Split (đếm số lần feature được dùng để chia nhánh) - phương pháp này có thể thiên vị features có nhiều giá trị liên tục. Ở đây nhóm so sánh trực tiếp với SHAP (phương pháp công bằng hơn về mặt lý thuyết) để xem có sự khác biệt lớn không.

In [ ]:
# 1. Xếp hạng theo SHAP
fi_shap = pd.Series(mean_abs_shap, index=X_test.columns).sort_values(ascending=False)
comparison_data = {"Feature": fi_shap.index, "SHAP_rank": range(1, len(fi_shap) + 1)}

# 2. Xếp hạng theo Model Gốc
fi_model = None

# Nếu là mô hình Tree-based
if hasattr(best_model, "feature_importances_"):
    fi_model = pd.Series(best_model.feature_importances_, index=X_test.columns)
# Nếu là mô hình Linear (như Ridge)
elif hasattr(best_model, "coef_"):
    # Lấy trị tuyệt đối của trọng số vì ta chỉ quan tâm độ lớn tác động
    fi_model = pd.Series(np.abs(best_model.coef_), index=X_test.columns)

# 3. Ghép kết quả và tính chênh lệch
if fi_model is not None:
    fi_model = fi_model.sort_values(ascending=False)
    model_rank_map = {feat: i + 1 for i, feat in enumerate(fi_model.index)}
    comparison_data["Model_rank"] = [model_rank_map.get(f, None) for f in fi_shap.index]

df_compare = pd.DataFrame(comparison_data).head(15)

# Tính chênh lệch thứ hạng
if "Model_rank" in df_compare.columns:
    df_compare["Rank_diff"] = (df_compare["Model_rank"] - df_compare["SHAP_rank"]).abs()

print("So sánh Top 15 Features: SHAP vs Feature Importance")
display(df_compare)

### **5.7. Kết luận SHAP**

| Bước | Kết quả |
|------|---------|
| Tính SHAP values | TreeExplainer/LinearExplainer trên tập Test |
| Summary Plot | Xác định feature quan trọng nhất |
| Beeswarm Plot | Xác định chiều hướng tác động |
| Waterfall Plot | Giải thích 1 dự đoán cụ thể (ngày AQI cao nhất) |
| Dependence Plot | Phát hiện tương tác giữa 2 features |
| So sánh với Split Importance | Xác nhận tính nhất quán giữa 2 phương pháp |

#### **Kết luận tổng thể:**
- **Top 3 features quan trọng nhất theo SHAP:**
1.  pm2_5: Yếu tố cốt lõi mang tính quyết định tuyệt đối đến dự đoán AQI.
2.  diff_1: Biến động lượng đóng vai trò hãm và điều chỉnh sai số.
3.  nitrogen_dioxide (NO2): Loại khí thải độc hại có sức nặng lớn thứ hai sau bụi mịn.

- **Sự đồng thuận đa chiều (SHAP vs. EDA vs. Model):**
1. Khớp với Model Coefficients: Có sự nhất quán gần như tuyệt đối (chênh lệch hạng = 0) ở 7 biến quan trọng nhất giữa phương pháp giải thích SHAP và trọng số toán học nội tại của mô hình Ridge. Điều này chứng minh mô hình học được các quy luật rất vững chắc, không bị nhiễu.

2. Khớp với phần khám phá dữ liệu (EDA): Phân tích đa biến của SHAP đã giải thích sâu hơn những gì EDA nhìn thấy. Trong EDA (đơn biến), ta thấy t-1 (AQI hôm qua) và mùa khô tỷ lệ thuận với mức độ ô nhiễm. Tuy nhiên, SHAP tiết lộ rằng trong mô hình dự đoán, một khi đã biết chính xác nồng độ pm2_5 của ngày hôm nay, mô hình sẽ dùng t-1 và các biến chu kỳ như hệ số bù trừ để tránh hiện tượng cộng dồn thông tin, giúp dự đoán không bị vọt lên lố đà. Sự tương tác mùa vụ (is_dry_season) cũng bị lu mờ hoàn toàn trước sức ảnh hưởng trực tiếp của nồng độ bụi.

- **Ý nghĩa thực tiễn (Sự minh bạch của mô hình):**
1. Phá vỡ Hộp đen (Black-box): Biểu đồ Waterfall có thể giải thích minh bạch đến từng điểm dự đoán cho bất kỳ ngày cụ thể nào: Bắt đầu từ mức trung bình là bao nhiêu, bị đẩy lên do đâu (như bụi mịn, NO2), và được kéo xuống nhờ yếu tố nào (như biến động lượng).

2. Tính logic và độ tin cậy: Biểu đồ Dependence Plot vẽ ra một đường thẳng hoàn hảo cho pm2_5, xác nhận Ridge Regression đang hoạt động cực kỳ ổn định, đúng với bản chất tuyến tính của nó. Hoàn toàn có thể tin tưởng rằng mô hình đưa ra dự đoán dựa trên các cơ sở khoa học về môi trường một cách hợp lý, chứ không phải học vẹt từ các điểm dữ liệu nhiễu.